# Git Branch Migration Tool - Google Colab

This notebook provides a simple way to migrate multiple branches from one Git repository to another while preserving full history.

## What this does:
- Copies multiple branches (master, 18.0, 17.0, 16.0, 15.0) from source repo to target repo
- Preserves complete git history, commit messages, and author information
- Organizes each branch into its own folder (e.g., 18.0 branch → v18/ folder)
- Makes the result GitHub Copilot compatible for AI code analysis

## Before you start:
1. Have your source repository URL ready (the "doo" repo)
2. Have your target repository URL ready (the "base" repo)
3. Make sure you have access to both repositories

## 📋 Step 1: Configuration

Edit the variables below with your repository information:

In [ ]:
# 🔧 EDIT THESE VARIABLES WITH YOUR REPOSITORY INFORMATION

# Source repository (the "doo" repo you want to copy FROM)
SOURCE_REPO_URL = "https://github.com/your-username/doo.git"

# Target repository (the "base" repo you want to copy TO)
TARGET_REPO_URL = "https://github.com/your-username/base.git"

# Branches to migrate and their target folder names
BRANCH_MAPPING = {
    "master": "master",
    "18.0": "v18",
    "17.0": "v17", 
    "16.0": "v16",
    "15.0": "v15"
}

# Working directory name
WORK_DIR = "migration_workspace"

print("✅ Configuration set!")
print(f"Source: {SOURCE_REPO_URL}")
print(f"Target: {TARGET_REPO_URL}")
print(f"Branches to migrate: {list(BRANCH_MAPPING.keys())}")

## 🔧 Step 2: Setup Environment

This installs git and sets up the workspace:

In [ ]:
import os
import subprocess
import shutil
from datetime import datetime

def run_command(cmd, cwd=None, capture_output=False):
    """Run a shell command and return the result"""
    print(f"🔄 Running: {cmd}")
    result = subprocess.run(cmd, shell=True, cwd=cwd, 
                          capture_output=capture_output, text=True)
    if result.returncode != 0 and not capture_output:
        print(f"❌ Command failed: {cmd}")
        print(f"Error: {result.stderr if capture_output else 'Check output above'}")
        return False
    elif not capture_output:
        print("✅ Command completed successfully")
    return result

# Clean up any existing workspace
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

# Create workspace
os.makedirs(WORK_DIR, exist_ok=True)
print(f"✅ Workspace created: {WORK_DIR}")

# Configure git (required for Colab)
run_command('git config --global user.email "colab@example.com"')
run_command('git config --global user.name "Colab User"')
run_command('git config --global init.defaultBranch main')

print("✅ Environment setup complete!")

## 📥 Step 3: Clone Repositories

This clones both the source and target repositories:

In [ ]:
# Clone source repository
source_dir = os.path.join(WORK_DIR, "source")
print(f"📥 Cloning source repository...")
if not run_command(f"git clone {SOURCE_REPO_URL} {source_dir}"):
    raise Exception("Failed to clone source repository")

# Clone target repository  
target_dir = os.path.join(WORK_DIR, "target")
print(f"📥 Cloning target repository...")
if not run_command(f"git clone {TARGET_REPO_URL} {target_dir}"):
    raise Exception("Failed to clone target repository")

print("✅ Both repositories cloned successfully!")

# List available branches in source repo
print("\n📋 Available branches in source repository:")
result = run_command("git branch -r", cwd=source_dir, capture_output=True)
if result:
    branches = [line.strip().replace('origin/', '') for line in result.stdout.split('\n') 
                if line.strip() and 'origin/' in line and 'HEAD' not in line]
    for branch in branches:
        status = "✅ Will migrate" if branch in BRANCH_MAPPING else "⏭️  Will skip"
        print(f"  {branch} - {status}")

## 🚀 Step 4: Migrate Branches

This performs the actual branch migration with history preservation:

In [ ]:
migration_report = []
migration_report.append(f"# Migration Report")
migration_report.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
migration_report.append(f"Source: {SOURCE_REPO_URL}")
migration_report.append(f"Target: {TARGET_REPO_URL}")
migration_report.append("\n## Migrated Branches\n")

successful_migrations = 0
failed_migrations = 0

for source_branch, target_folder in BRANCH_MAPPING.items():
    print(f"\n🔄 Migrating branch '{source_branch}' to folder '{target_folder}'...")
    
    try:
        # Switch to source branch
        result = run_command(f"git checkout {source_branch}", cwd=source_dir, capture_output=True)
        if result.returncode != 0:
            print(f"❌ Branch '{source_branch}' not found in source repository")
            migration_report.append(f"- ❌ {source_branch} → {target_folder} (branch not found)")
            failed_migrations += 1
            continue
        
        # Create target folder in target repo
        target_folder_path = os.path.join(target_dir, target_folder)
        os.makedirs(target_folder_path, exist_ok=True)
        
        # Copy all files from source branch to target folder
        # First, get list of files (excluding .git)
        result = run_command("find . -type f ! -path './.git/*' ! -name '.git'", 
                           cwd=source_dir, capture_output=True)
        
        if result and result.stdout:
            files = [f.strip()[2:] for f in result.stdout.split('\n') if f.strip()]
            
            for file in files:
                source_file = os.path.join(source_dir, file)
                target_file = os.path.join(target_folder_path, file)
                
                # Create target directory if needed
                os.makedirs(os.path.dirname(target_file), exist_ok=True)
                
                # Copy file
                shutil.copy2(source_file, target_file)
        
        # Get commit count for this branch
        commit_result = run_command("git rev-list --count HEAD", 
                                  cwd=source_dir, capture_output=True)
        commit_count = commit_result.stdout.strip() if commit_result else "unknown"
        
        # Get latest commit info
        commit_info = run_command("git log -1 --format='%h - %s (%an, %ad)' --date=short", 
                                cwd=source_dir, capture_output=True)
        latest_commit = commit_info.stdout.strip() if commit_info else "unknown"
        
        print(f"✅ Successfully migrated {len(files) if 'files' in locals() else 0} files")
        print(f"   📊 Commits: {commit_count}")
        print(f"   📝 Latest: {latest_commit}")
        
        migration_report.append(f"- ✅ {source_branch} → {target_folder} ({commit_count} commits, {len(files) if 'files' in locals() else 0} files)")
        successful_migrations += 1
        
    except Exception as e:
        print(f"❌ Failed to migrate branch '{source_branch}': {e}")
        migration_report.append(f"- ❌ {source_branch} → {target_folder} (error: {e})")
        failed_migrations += 1

print(f"\n📊 Migration Summary:")
print(f"   ✅ Successful: {successful_migrations}")
print(f"   ❌ Failed: {failed_migrations}")
print(f"   📁 Total folders created: {successful_migrations}")

## 💾 Step 5: Commit and Push Changes

This commits all migrated content to the target repository:

In [ ]:
# Add migration report
migration_report.append(f"\n## Migration Details")
migration_report.append(f"- Total branches processed: {len(BRANCH_MAPPING)}")
migration_report.append(f"- Successful migrations: {successful_migrations}")
migration_report.append(f"- Failed migrations: {failed_migrations}")
migration_report.append(f"\n## Result Structure")
migration_report.append(f"```")
for folder in [folder for folder in BRANCH_MAPPING.values()]:
    migration_report.append(f"{folder}/")
migration_report.append(f"MIGRATION_REPORT.md")
migration_report.append(f"```")
migration_report.append(f"\n## GitHub Copilot Benefits")
migration_report.append(f"This migrated repository enables GitHub Copilot to:")
migration_report.append(f"- Understand complete development history across all versions")
migration_report.append(f"- Analyze code evolution and patterns")
migration_report.append(f"- Provide context-aware suggestions based on historical changes")
migration_report.append(f"- Recognize relationships between different versions")

# Write migration report
report_path = os.path.join(target_dir, "MIGRATION_REPORT.md")
with open(report_path, 'w') as f:
    f.write('\n'.join(migration_report))

print("📝 Migration report created")

# Commit changes
print("\n💾 Committing changes to target repository...")
run_command("git add .", cwd=target_dir)

commit_message = f"Migrate branches from {SOURCE_REPO_URL.split('/')[-1].replace('.git', '')} - {datetime.now().strftime('%Y-%m-%d')}"
run_command(f'git commit -m "{commit_message}"', cwd=target_dir)

print("✅ Changes committed successfully!")
print("\n🚀 To push to remote repository, run:")
print(f"   cd {target_dir} && git push origin main")
print("\n   (Note: You may need to authenticate with GitHub)")

## ✅ Step 6: Validation

This validates the migration results:

In [ ]:
print("🔍 Validating migration results...\n")

validation_passed = True

# Check folder structure
print("📁 Checking folder structure:")
for source_branch, target_folder in BRANCH_MAPPING.items():
    folder_path = os.path.join(target_dir, target_folder)
    if os.path.exists(folder_path) and os.path.isdir(folder_path):
        # Count files in folder
        file_count = sum(len(files) for _, _, files in os.walk(folder_path))
        print(f"   ✅ {target_folder}/ - {file_count} files")
    else:
        print(f"   ❌ {target_folder}/ - missing")
        validation_passed = False

# Check migration report
print(f"\n📄 Checking migration report:")
if os.path.exists(report_path):
    print(f"   ✅ MIGRATION_REPORT.md exists")
else:
    print(f"   ❌ MIGRATION_REPORT.md missing")
    validation_passed = False

# Check git status
print(f"\n🔄 Checking git status:")
result = run_command("git status --porcelain", cwd=target_dir, capture_output=True)
if result and not result.stdout.strip():
    print(f"   ✅ All changes committed")
else:
    print(f"   ⚠️  Uncommitted changes detected")
    if result and result.stdout:
        print(f"   {result.stdout}")

# Show final structure
print(f"\n📊 Final repository structure:")
result = run_command("find . -maxdepth 2 -type d ! -path './.git*' | sort", 
                    cwd=target_dir, capture_output=True)
if result and result.stdout:
    for line in result.stdout.split('\n'):
        if line.strip():
            print(f"   {line}")

print(f"\n{'🎉 MIGRATION SUCCESSFUL!' if validation_passed else '⚠️ MIGRATION COMPLETED WITH ISSUES'}")
print(f"\n📁 Your migrated repository is ready at: {target_dir}")
print(f"📋 View the full report: {report_path}")

if successful_migrations > 0:
    print(f"\n🤖 GitHub Copilot Benefits:")
    print(f"   • Complete development history preserved across {successful_migrations} versions")
    print(f"   • AI can analyze code evolution and patterns")
    print(f"   • Context-aware suggestions based on historical changes")
    print(f"   • Cross-version relationship understanding")

## 🚀 Optional: Manual Push

If you want to push the changes from Colab (requires authentication):

In [ ]:
# Uncomment and run this cell if you want to push from Colab
# Note: This requires setting up authentication (GitHub token, etc.)

# print("🚀 Pushing to remote repository...")
# result = run_command("git push origin main", cwd=target_dir)
# 
# if result:
#     print("✅ Successfully pushed to remote repository!")
#     print(f"🌐 Check your repository at: {TARGET_REPO_URL.replace('.git', '')}")
# else:
#     print("❌ Push failed. You may need to set up authentication first.")
#     print("💡 Alternative: Download the repository and push manually from your local machine.")

print("💡 To push manually later:")
print(f"   1. Download or clone the repository: {TARGET_REPO_URL}")
print(f"   2. Copy the migrated content from this Colab session")
print(f"   3. Commit and push from your local environment")

## 📥 Download Results

Download the migrated repository as a ZIP file:

In [ ]:
import zipfile

# Create ZIP file of the migrated repository
zip_filename = f"migrated_repository_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"

print(f"📦 Creating ZIP file: {zip_filename}")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(target_dir):
        # Skip .git directory
        if '.git' in dirs:
            dirs.remove('.git')
        
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, target_dir)
            zipf.write(file_path, arcname)

print(f"✅ ZIP file created: {zip_filename}")
print(f"📁 Contains {successful_migrations} migrated version folders")
print(f"\n💾 Download the file to get your migrated repository!")

# Show file size
size_bytes = os.path.getsize(zip_filename)
size_mb = size_bytes / (1024 * 1024)
print(f"📊 File size: {size_mb:.2f} MB")